# Kerendia LGBM Feature Selection - PyArrow + Native LightGBM + MLflow Logging

This notebook keeps the original feature-selection flow, and adds:

- MLflow experiment tracking for safe parameters, metrics, model artifacts, SHAP plots, PDP plots, and selected-feature outputs
- SHAP beeswarm and mean absolute SHAP plots for top 100, 200, 500, 1000, all content features, and content-in-top-1000
- PDP plots for content features only


## Cell 1 - Install packages
Run once if the environment does not already have these libraries. Databricks will restart Python after installation.

In [0]:
# Databricks only. Run once if packages are missing.
%pip install -U lightgbm pyarrow pandas numpy scikit-learn psutil mlflow shap

# Restart Python after %pip installs in Databricks.
dbutils.library.restartPython()

## Cell 2 - Imports

In [0]:
import os
import gc
import json
import time
import shutil
import tempfile
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import lightgbm as lgb
import psutil

from sklearn.metrics import r2_score

import mlflow
import mlflow.lightgbm
import shap
import matplotlib.pyplot as plt

## Cell 3 - Config
Update `DATA_PATH`, `OUTPUT_DIR`, and `EXCLUDE_COLS` as needed. Feature-penalty details are used internally only and are not logged, saved, or printed.

In [0]:
# ============================================================
# Input / output config
# ============================================================

DATA_PATH = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/"
    "final_dataset_parquet"
)

OUTPUT_DIR = (
    "/Volumes/"
    "ph-com-ai-us-prod-catalog-2553213839599380/"
    "kerendia_oe_journey/"
    "lgbm_feature_selection/v2/"
)

try:
    dbutils.fs.mkdirs(f"dbfs:{OUTPUT_DIR}")
except Exception:
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

LOCAL_OUTPUT_DIR = "/tmp/lgbm_feature_selection_logging_outputs"
Path(LOCAL_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ============================================================
# Column config
# ============================================================

TARGET_COL = "TOTAL_NBRX_C4"
ID_COL = "PROVIDER_ID"

# Original exclusion list from the base notebook.
EXCLUDE_COLS = {
    TARGET_COL,
    "PROVIDER_ID", "E_FIELD_ADDED_NOT_SEGMENTED","D_LOWER_POTENTIAL_NON_WRITER_SEGMENT","UNASSIGNED_SEGMENT",'ADVISORY_BOARD_ATTENDED_R_P3', 'DOCEREE_BANNER_IMPRESSION_R_P3', 'DOXIMITY_FORMULARYONE_VIEW_R_P3', 'EPOCRATES_DOCALERT_DOCALERT_DELIVERED_R_P3', 'EPOCRATES_MONOGRAPH_MESSAGE_MONOGRAPH_MESSAGE_HEADLINE_IMPRESSION_R_P3', 'EPOCRATES_SEARCH_SUGGESTIONS_SEARCH_SUGGESTION_HEADLINE_IMPRESSION_R_P3', 'IMPIRICUS_MOBILE_SEND_R_P3', 'INVESTIGATOR_MEETING_ATTENDED_R_P3', 'MARKET_ACCESS_FORMULARY_ASSIST_ATTENDED_R_P3', 'MD_CALC_BRANDALERT_ENGAGEMENT_R_P3', 'MEDSCAPE_BRANDSPOT_RICH_MEDIA_ENGAGEMENT_R_P3', 'MEDSCAPE_EHR_EXPOSURE_R_P3', 'MEDSCAPE_FFQ_PAGE_VIEW_R_P3', 'MEDSCAPE_RECAP_PAGE_VIEW_R_P3', 'MEDSCAPE_XCOD_VIEW_R_P3', 'MJHLIFESCIENCES_3RD_PARTY_EMAIL_SENT_R_P3', 'MJHLIFESCIENCES_BANNER_IMPRESSIONS_R_P3', 'NATIVO_BANNER_IMPRESSIONS_R_P3', 'PEERDIRECT_FFQ_VIEW_R_P3', 'PEERDIRECT_MDFI_WEB_VIEW_R_P3', 'PRESCRIBER_POINT_CUSTOM_IMPRESSION_R_P3', 'REACHMD_DESKTOP_PAGE_VIEW_R_P3', 'REACHMD_EMAIL_EMAIL_DELIVERED_R_P3', 'SCIENTIFIC_PRESENTATION_ATTENDED_R_P3', 'SCIENTIFIC_ROUND_TABLES_ATTENDED_R_P3', 'SEARCH_BANNER_CLICKED_R_P3', 'SPONSORSHIP_ATTENDED_R_P3', 'SYMPOSIUM_ATTENDED_R_P3', 'ADVISORY_BOARD_ATTENDED_I_P3', 'DOCEREE_BANNER_CLICK_I_P3', 'DOXIMITY_FORMULARYONE_CLICK_THROUGH_I_P3', 'EPOCRATES_DOCALERT_DOCALERT_OPENED_I_P3', 'EPOCRATES_MONOGRAPH_MESSAGE_MONOGRAPH_MESSAGE_PRESCRIBING_INFORMATION_CLICKED_I_P3', 'EPOCRATES_SEARCH_SUGGESTIONS_SEARCH_SUGGESTION_HEADLINE_CLICKED_I_P3', 'IMPIRICUS_MOBILE_CLICK_I_P3', 'INVESTIGATOR_MEETING_ATTENDED_I_P3', 'MARKET_ACCESS_FORMULARY_ASSIST_ATTENDED_I_P3', 'MD_CALC_BRANDALERT_CLICK_I_P3', 'MEDSCAPE_BRANDSPOT_RICH_MEDIA_VIDEO_TIME_SPENT_I_P3', 'MEDSCAPE_FFQ_LINK_CLICK_I_P3', 'MEDSCAPE_RECAP_VIDEO_TIME_SPENT_I_P3', 'MEDSCAPE_XCOD_MEDIA_TIME_SPENT_I_P3', 'MJHLIFESCIENCES_3RD_PARTY_EMAIL_OPEN_I_P3', 'NATIVO_BANNER_CLICKS_I_P3', 'PEERDIRECT_FFQ_CLICK_I_P3', 'PRESCRIBER_POINT_CUSTOM_VISIT_I_P3', 'REACHMD_DESKTOP_PARTICIPATION_I_P3', 'REACHMD_EMAIL_EMAIL_OPEN_I_P3', 'SCIENTIFIC_PRESENTATION_ATTENDED_I_P3', 'SCIENTIFIC_ROUND_TABLES_ATTENDED_I_P3', 'SEARCH_BANNER_CLICKED_I_P3', 'SPONSORSHIP_ATTENDED_I_P3', 'SYMPOSIUM_ATTENDED_I_P3', 'ADVISORY_BOARD_ATTENDED_E_P3', 'DOCEREE_BANNER_CLICK_E_P3', 'DOXIMITY_FORMULARYONE_CLICK_THROUGH_E_P3', 'EPOCRATES_DOCALERT_DOCALERT_OPENED_E_P3', 'EPOCRATES_MONOGRAPH_MESSAGE_MONOGRAPH_MESSAGE_PRESCRIBING_INFORMATION_CLICKED_E_P3', 'EPOCRATES_SEARCH_SUGGESTIONS_SEARCH_SUGGESTION_HEADLINE_CLICKED_E_P3', 'IMPIRICUS_MOBILE_CLICK_E_P3', 'INVESTIGATOR_MEETING_ATTENDED_E_P3', 'MARKET_ACCESS_FORMULARY_ASSIST_ATTENDED_E_P3', 'MD_CALC_BRANDALERT_CLICK_E_P3', 'MEDSCAPE_BRANDSPOT_RICH_MEDIA_LINK_CLICK_E_P3', 'MEDSCAPE_FFQ_RESPONSE_E_P3', 'MEDSCAPE_RECAP_PARTICIPATION_E_P3', 'MEDSCAPE_XCOD_LINK_CLICK_E_P3', 'MJHLIFESCIENCES_3RD_PARTY_EMAIL_CLICK_E_P3', 'NATIVO_BANNER_CLICKS_E_P3', 'SCIENTIFIC_PRESENTATION_ATTENDED_E_P3', 'SCIENTIFIC_ROUND_TABLES_ATTENDED_E_P3', 'SEARCH_BANNER_CLICKED_E_P3', 'SPONSORSHIP_ATTENDED_E_P3', 'SYMPOSIUM_ATTENDED_E_P3','ADDITIONAL_DATA_COUNT_P3', 'BRAND_EXPERIENCE_COUNT_P3', 'EFFICACY_COUNT_P3', 'EVENT_COUNT_P3', 'MECHANISM_OF_DISEASE_COUNT_P3', 'MISCELLANEOUS_COUNT_P3', 'PATIENT_ADHERENCE_SUPPORT_COUNT_P3', 'SOCIAL_PROOF_COUNT_P3',"MEDSCAPE_PULSEPOINT_BANNER_PULSEPOINT_MEDIA_IMPRESSION_R_P3","MEDSCAPE_BANNER_PULSEPOINT_MEDIA_IMPRESSION_R_P3","MEDSCAPE_PULSEPOINT_BANNER_PULSEPOINT_MEDIA_CLICK_I_P3","MEDSCAPE_PULSEPOINT_BANNER_PULSEPOINT_MEDIA_CLICK_E_P3","MEDSCAPE_BANNER_PULSEPOINT_MEDIA_CLICK_E_P3","MEDSCAPE_BANNER_PULSEPOINT_MEDIA_CLICK_I_P3","GENERAL_COUNT_P3","DISEASE_AWARENESS_MANAGEMENT_COUNT_P3","GUIDELINES_COUNT_P3","UNMET_NEED_COUNT_P3","PATIENT_PROFILE_COUNT_P3","COST_COUNT_P3","BASIC_PRODUCT_INFO_COUNT_P3","SAMPLES_COUNT_P3","ACCESS_COVERAGE_COUNT_P3","STUDY_DESIGN_COUNT_P3","MECHANISM_OF_ACTION_COUNT_P3","DOSING_ADMINISTRATION_COUNT_P3","SAFETY_EFFICACY_COUNT_P3"  
}

# Keep ID_COL aligned with the original exclusion list.
EXCLUDE_COLS.add(ID_COL)

# ============================================================
# Train/test config
# ============================================================

TRAIN_RATIO = 0.80
SEED = 42

# For quick testing set MAX_ROWS_TO_USE = 1000.
# For full run, keep None.
MAX_ROWS_TO_USE = None

# ============================================================
# LightGBM config
# ============================================================

TOP_K = 1000

NUM_ITERATIONS = 250
LEARNING_RATE = 0.03
NUM_LEAVES = 31
MIN_DATA_IN_LEAF = 500

FEATURE_FRACTION = 0.8
BAGGING_FRACTION = 0.8
BAGGING_FREQ = 5

MAX_BIN = 127
BIN_CONSTRUCT_SAMPLE_CNT = 200000

LAMBDA_L1 = 1
MIN_GAIN_TO_SPLIT = 0.01

EARLY_STOPPING_ROUNDS = 30

# Internal feature penalty map.
# IMPORTANT: this is NOT logged, saved, or printed by this notebook.
_FEATURE_PENALTY_MAP = {
    "TOTAL_NBRX_P3": 0.2,
    "TOTAL_TRX_P3": 0.2,
    "PD_TRX_P3": 0.2,
}

# ============================================================
# Thread config
# ============================================================

NUM_THREADS = psutil.cpu_count(logical=False)
if NUM_THREADS is None:
    NUM_THREADS = 64
NUM_THREADS = min(NUM_THREADS, 64)

# ============================================================
# Data loading config
# ============================================================

ARROW_BATCH_SIZE = 2048
ARRAY_DTYPE = np.float32

# ============================================================
# SHAP config
# ============================================================

SHAP_BATCH_SIZE = 512
PLOT_ROW_SAMPLE = 10000

# HCP-level SHAP output.
# Default is top-1000 SHAP features to keep file size manageable.
# Set HCP_SHAP_USE_ALL_FEATURES=True only if you truly need all feature SHAP columns per HCP.
HCP_SHAP_USE_ALL_FEATURES = False
HCP_SHAP_BATCH_SIZE = 2000

# ============================================================
# Content feature prefixes
# ============================================================

CONTENT_PREFIXES = [
    "GENERAL_COUNT",
    "PATIENT_PROFILE",
    "SAMPLES",
    "SAFETY_EFFICACY",
    "BASIC_PRODUCT_INFO",
    "DISEASE_AWARENESS_MANAGEMENT",
    "STUDY_DESIGN",
    "MECHANISM_OF_ACTION",
    "UNMET_NEED",
    "ACCESS_COVERAGE",
    "DOSING_ADMINISTRATION",
    "COST",
    "GUIDELINES"
]

# PDP for content features.
# Set CONTENT_PDP_ROWS=None for all train rows; 50k is much faster and usually stable.
CONTENT_PDP_ROWS = None
CONTENT_PDP_GRID_SIZE = 20
CONTENT_PDP_BATCH_SIZE = 2000
CONTENT_PDP_MAX_FEATURES = None

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("NUM_THREADS:", NUM_THREADS)

## Cell 4 - MLflow setup and safe save helpers
This logs parameters and artifacts while intentionally skipping feature-penalty details.

In [0]:
# ============================================================
# MLflow setup
# ============================================================

try:
    db_user = (
        dbutils.notebook
        .entry_point
        .getDbutils()
        .notebook()
        .getContext()
        .userName()
        .get()
    )
except Exception:
    db_user = "shared"

MLFLOW_EXPERIMENT_NAME = f"/Users/{db_user}/kerendia_lgbm_feature_selection_pyarrow"
MLFLOW_RUN_NAME = f"lgbm_pyarrow_feature_selection_{time.strftime('%Y%m%d_%H%M%S')}"

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

if mlflow.active_run() is not None:
    mlflow.end_run()

active_run = mlflow.start_run(run_name=MLFLOW_RUN_NAME)
print("MLflow experiment:", MLFLOW_EXPERIMENT_NAME)
print("MLflow run id:", active_run.info.run_id)

# ============================================================
# Helper functions
# ============================================================

DO_NOT_LOG_PARAM_KEYS = {"feature_penalty"}


def safe_filename(x):
    import re
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(x))[:180]


def copy_local_to_output(local_path, output_rel_path=None):
    local_path = str(local_path)
    if output_rel_path is None:
        output_rel_path = os.path.basename(local_path)

    output_dir_clean = OUTPUT_DIR.rstrip("/")
    final_output_path = f"{output_dir_clean}/{output_rel_path}"

    try:
        final_dir = os.path.dirname(final_output_path)
        dbutils.fs.mkdirs(f"dbfs:{final_dir}")
        dbutils.fs.cp(f"file:{local_path}", f"dbfs:{final_output_path}", True)
    except Exception:
        Path(os.path.dirname(final_output_path)).mkdir(parents=True, exist_ok=True)
        shutil.copyfile(local_path, final_output_path)

    print("Saved:", final_output_path)
    return final_output_path


def log_artifact_if_active(local_path, artifact_path=None):
    if mlflow.active_run() is not None:
        mlflow.log_artifact(local_path, artifact_path=artifact_path)


def save_df_to_output(df, filename, log_to_mlflow=True, artifact_path=None):
    local_path = os.path.join(LOCAL_OUTPUT_DIR, os.path.basename(filename))
    df.to_csv(local_path, index=False)
    output_path = copy_local_to_output(local_path, filename)
    if log_to_mlflow:
        log_artifact_if_active(local_path, artifact_path=artifact_path)
    return output_path


def save_json_to_output(obj, filename, log_to_mlflow=True, artifact_path=None):
    local_path = os.path.join(LOCAL_OUTPUT_DIR, os.path.basename(filename))
    with open(local_path, "w") as f:
        json.dump(obj, f, indent=2, default=str)
    output_path = copy_local_to_output(local_path, filename)
    if log_to_mlflow:
        log_artifact_if_active(local_path, artifact_path=artifact_path)
    return output_path


def save_text_list_to_output(values, filename, log_to_mlflow=True, artifact_path=None):
    local_path = os.path.join(LOCAL_OUTPUT_DIR, os.path.basename(filename))
    with open(local_path, "w") as f:
        for v in values:
            f.write(str(v) + "\n")
    output_path = copy_local_to_output(local_path, filename)
    if log_to_mlflow:
        log_artifact_if_active(local_path, artifact_path=artifact_path)
    return output_path


def save_current_fig(filename, log_to_mlflow=True, artifact_path=None, dpi=300):
    local_path = os.path.join(LOCAL_OUTPUT_DIR, os.path.basename(filename))
    plt.savefig(local_path, dpi=dpi, bbox_inches="tight")
    output_path = copy_local_to_output(local_path, filename)
    if log_to_mlflow:
        log_artifact_if_active(local_path, artifact_path=artifact_path)
    return output_path


def safe_log_params(params, prefix=""):
    if mlflow.active_run() is None:
        return
    for k, v in params.items():
        if k in DO_NOT_LOG_PARAM_KEYS:
            continue
        try:
            key = f"{prefix}{k}"
            if isinstance(v, (list, tuple, dict, set)):
                value = json.dumps(list(v) if isinstance(v, set) else v, default=str)[:500]
            else:
                value = str(v)[:500]
            mlflow.log_param(key, value)
        except Exception as e:
            print(f"Skipping MLflow param {k}: {e}")


def safe_log_metrics(metrics, prefix=""):
    if mlflow.active_run() is None:
        return
    for k, v in metrics.items():
        try:
            if v is not None and np.isfinite(v):
                mlflow.log_metric(f"{prefix}{k}", float(v))
        except Exception as e:
            print(f"Skipping MLflow metric {k}: {e}")


def is_content_feature(feature_name):
    return any(str(feature_name).startswith(prefix) for prefix in CONTENT_PREFIXES)


safe_log_params({
    "target_col": TARGET_COL,
    "id_col": ID_COL,
    "train_ratio": TRAIN_RATIO,
    "seed": SEED,
    "top_k": TOP_K,
    "num_iterations": NUM_ITERATIONS,
    "learning_rate": LEARNING_RATE,
    "num_leaves": NUM_LEAVES,
    "min_data_in_leaf": MIN_DATA_IN_LEAF,
    "feature_fraction": FEATURE_FRACTION,
    "bagging_fraction": BAGGING_FRACTION,
    "bagging_freq": BAGGING_FREQ,
    "max_bin": MAX_BIN,
    "bin_construct_sample_cnt": BIN_CONSTRUCT_SAMPLE_CNT,
    "lambda_l1": LAMBDA_L1,
    "min_gain_to_split": MIN_GAIN_TO_SPLIT,
    "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
    "num_threads": NUM_THREADS,
    "arrow_batch_size": ARROW_BATCH_SIZE,
    "shap_batch_size": SHAP_BATCH_SIZE,
    "plot_row_sample": PLOT_ROW_SAMPLE,
    "hcp_shap_use_all_features": HCP_SHAP_USE_ALL_FEATURES,
    "content_pdp_rows": CONTENT_PDP_ROWS,
    "content_pdp_grid_size": CONTENT_PDP_GRID_SIZE,
})

save_json_to_output(
    {"content_prefixes": CONTENT_PREFIXES},
    "run_content_prefixes.json",
    log_to_mlflow=True,
    artifact_path="run_config"
)

## Cell 5 - Machine check

In [0]:
print("===================================================")
print("MACHINE CHECK")
print("===================================================")

vm = psutil.virtual_memory()

print(f"CPU logical cores       : {psutil.cpu_count(logical=True)}")
print(f"CPU physical cores      : {psutil.cpu_count(logical=False)}")
print(f"LightGBM NUM_THREADS    : {NUM_THREADS}")
print(f"Total memory GiB        : {vm.total / (1024 ** 3):,.2f}")
print(f"Available memory GiB    : {vm.available / (1024 ** 3):,.2f}")
print("===================================================")

safe_log_metrics({
    "machine_cpu_logical_cores": psutil.cpu_count(logical=True),
    "machine_cpu_physical_cores": psutil.cpu_count(logical=False) or NUM_THREADS,
    "machine_total_memory_gib": vm.total / (1024 ** 3),
    "machine_available_memory_gib": vm.available / (1024 ** 3)
})

## Cell 6 - Read Parquet schema and select features

In [0]:
# ============================================================
# Read Parquet dataset
# ============================================================

dataset = ds.dataset(DATA_PATH, format="parquet")
schema = dataset.schema
all_cols = schema.names

if TARGET_COL not in all_cols:
    raise ValueError(f"Target column '{TARGET_COL}' not found in dataset.")

if ID_COL not in all_cols:
    raise ValueError(f"ID column '{ID_COL}' not found in dataset. Required for HCP-level SHAP output.")


def is_arrow_numeric_type(dtype):
    return (
        pa.types.is_integer(dtype)
        or pa.types.is_floating(dtype)
        or pa.types.is_decimal(dtype)
        or pa.types.is_boolean(dtype)
    )

# ============================================================
# Select numeric feature columns
# ============================================================

feature_cols = []

for field in schema:
    col_name = field.name
    dtype = field.type

    if col_name in EXCLUDE_COLS:
        continue

    if is_arrow_numeric_type(dtype):
        feature_cols.append(col_name)

n_features = len(feature_cols)

if n_features == 0:
    raise ValueError("No numeric/decimal feature columns found.")

# Internal feature penalty vector. Do not print, save, or log.
feature_penalty = [_FEATURE_PENALTY_MAP.get(col, 1.0) for col in feature_cols]

print("Total columns in source dataset :", len(all_cols))
print("Numeric feature columns used    :", n_features)
print("Target column                   :", TARGET_COL)
print("ID column                       :", ID_COL)
print("Sample features:")
print(feature_cols[:20])

safe_log_metrics({
    "source_total_columns": len(all_cols),
    "numeric_feature_count": n_features
})

## Cell 7 - Feature alias mapping

In [0]:
feature_aliases = [f"f_{i}" for i in range(n_features)]

feature_mapping_df = pd.DataFrame({
    "feature_index": np.arange(n_features, dtype=np.int32),
    "feature_alias": feature_aliases,
    "feature": feature_cols
})

save_df_to_output(
    feature_mapping_df,
    "feature_alias_mapping_wo_segment_new_P12_fts.csv",
    log_to_mlflow=True,
    artifact_path="feature_metadata"
)

feature_mapping_df.head()

## Cell 8 - Count rows and create train/test split

In [0]:
n_rows_total = dataset.count_rows()

if MAX_ROWS_TO_USE is None:
    n_rows = n_rows_total
else:
    n_rows = min(MAX_ROWS_TO_USE, n_rows_total)

print("Total rows in parquet:", n_rows_total)
print("Rows used in this run:", n_rows)
print("Total features:", n_features)

rng = np.random.default_rng(SEED)
train_mask_all = rng.random(n_rows) < TRAIN_RATIO

n_train = int(train_mask_all.sum())
n_test = int(n_rows - n_train)

print("Train rows:", n_train)
print("Test rows :", n_test)
print("Train ratio actual:", n_train / n_rows)

safe_log_metrics({
    "data_n_rows_total": n_rows_total,
    "data_n_rows_used": n_rows,
    "data_n_train": n_train,
    "data_n_test": n_test,
    "data_train_ratio_actual": n_train / n_rows,
    "data_n_features": n_features
})

## Cell 9 - Allocate arrays

In [0]:
print("Allocating train/test arrays...")

start_time = time.time()

X_train = np.empty(shape=(n_train, n_features), dtype=ARRAY_DTYPE, order="C")
X_test = np.empty(shape=(n_test, n_features), dtype=ARRAY_DTYPE, order="C")
y_train = np.empty(shape=(n_train,), dtype=ARRAY_DTYPE)
y_test = np.empty(shape=(n_test,), dtype=ARRAY_DTYPE)

provider_train = np.empty(n_train, dtype=object)
provider_test = np.empty(n_test, dtype=object)

elapsed = time.time() - start_time

print(f"Allocation complete in {elapsed:,.2f} seconds")
print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("provider_train shape:", provider_train.shape)
print("provider_test shape :", provider_test.shape)

safe_log_metrics({"array_allocation_elapsed_seconds": elapsed})

## Cell 10 - Load Parquet into NumPy arrays and provider arrays
This reads `PROVIDER_ID`, target, and model features in one pass.

In [0]:
# ============================================================
# Columns to scan
# ============================================================

scan_cols = [ID_COL, TARGET_COL] + feature_cols
numeric_scan_cols = [TARGET_COL] + feature_cols

cast_schema = pa.schema([(col, pa.float32()) for col in numeric_scan_cols])

scanner = dataset.scanner(
    columns=scan_cols,
    batch_size=ARROW_BATCH_SIZE,
    use_threads=True
)

global_row_pos = 0
train_write_pos = 0
test_write_pos = 0

start_time = time.time()

for batch_id, record_batch in enumerate(scanner.to_batches(), start=1):
    batch_n = record_batch.num_rows

    if global_row_pos >= n_rows:
        break

    if global_row_pos + batch_n > n_rows:
        batch_n = n_rows - global_row_pos
        record_batch = record_batch.slice(0, batch_n)

    table = pa.Table.from_batches([record_batch])

    # ID values are kept separately and are not cast to float.
    id_values = table.column(ID_COL).to_pandas().to_numpy()

    numeric_table = table.select(numeric_scan_cols)
    numeric_table = numeric_table.cast(cast_schema, safe=False)

    pdf_batch = numeric_table.to_pandas(split_blocks=True, self_destruct=True)
    batch_arr = pdf_batch.to_numpy(dtype=np.float32, copy=True)

    y_batch = batch_arr[:, 0]
    bad_y_mask = ~np.isfinite(y_batch)
    if bad_y_mask.any():
        y_batch[bad_y_mask] = 0.0

    X_batch = batch_arr[:, 1:]
    bad_x_mask = np.isinf(X_batch)
    if bad_x_mask.any():
        X_batch[bad_x_mask] = np.nan

    batch_train_mask = train_mask_all[global_row_pos : global_row_pos + batch_n]
    batch_test_mask = ~batch_train_mask

    n_train_batch = int(batch_train_mask.sum())
    n_test_batch = int(batch_test_mask.sum())

    if n_train_batch > 0:
        X_train[train_write_pos : train_write_pos + n_train_batch, :] = X_batch[batch_train_mask, :]
        y_train[train_write_pos : train_write_pos + n_train_batch] = y_batch[batch_train_mask]
        provider_train[train_write_pos : train_write_pos + n_train_batch] = id_values[batch_train_mask]
        train_write_pos += n_train_batch

    if n_test_batch > 0:
        X_test[test_write_pos : test_write_pos + n_test_batch, :] = X_batch[batch_test_mask, :]
        y_test[test_write_pos : test_write_pos + n_test_batch] = y_batch[batch_test_mask]
        provider_test[test_write_pos : test_write_pos + n_test_batch] = id_values[batch_test_mask]
        test_write_pos += n_test_batch

    global_row_pos += batch_n

    if batch_id % 25 == 0 or global_row_pos == n_rows:
        elapsed = time.time() - start_time
        print(
            f"Batch {batch_id} | rows loaded: {global_row_pos:,} / {n_rows:,} "
            f"| elapsed sec: {elapsed:,.1f}"
        )

    del table, numeric_table, pdf_batch, batch_arr, X_batch, y_batch, id_values
    gc.collect()

elapsed = time.time() - start_time

print("===================================================")
print("LOAD COMPLETE")
print("===================================================")
print(f"Rows loaded           : {global_row_pos:,}")
print(f"Train rows written    : {train_write_pos:,}")
print(f"Test rows written     : {test_write_pos:,}")
print(f"Elapsed seconds       : {elapsed:,.2f}")
print("===================================================")

if train_write_pos != n_train:
    raise ValueError(f"Train row mismatch: expected {n_train}, got {train_write_pos}")

if test_write_pos != n_test:
    raise ValueError(f"Test row mismatch: expected {n_test}, got {test_write_pos}")

safe_log_metrics({"data_load_elapsed_seconds": elapsed})

## Cell 11 - Target checks

In [0]:
def summarize_target(y, name):
    total = len(y)
    zero_count = int(np.sum(y == 0))
    bad_count = int(np.sum(~np.isfinite(y)))

    out = {
        "rows": total,
        "zero_count": zero_count,
        "bad_count": bad_count,
        "min": float(np.nanmin(y)),
        "max": float(np.nanmax(y)),
        "mean": float(np.nanmean(y))
    }

    print(f"========== {name} ==========")
    for k, v in out.items():
        print(f"{k}: {v}")
    print()

    return out

train_target_summary = summarize_target(y_train, "TRAIN TARGET")
test_target_summary = summarize_target(y_test, "TEST TARGET")

safe_log_metrics(train_target_summary, prefix="target_train_")
safe_log_metrics(test_target_summary, prefix="target_test_")

if np.nanmin(y_train) < 0 or np.nanmin(y_test) < 0:
    raise ValueError("Poisson objective requires non-negative target values.")

## Cell 12 - Create LightGBM Dataset objects

In [0]:
dataset_params = {
    "max_bin": MAX_BIN,
    "bin_construct_sample_cnt": BIN_CONSTRUCT_SAMPLE_CNT,
    "feature_pre_filter": False
}

train_data = lgb.Dataset(
    data=X_train,
    label=y_train,
    feature_name=feature_aliases,
    free_raw_data=False,
    params=dataset_params
)

test_data = lgb.Dataset(
    data=X_test,
    label=y_test,
    feature_name=feature_aliases,
    reference=train_data,
    free_raw_data=False,
    params=dataset_params
)

print("LightGBM Dataset objects created.")

## Cell 13 - Train LightGBM with MLflow logging
Feature penalty is used in training but not logged/saved/printed.

In [0]:
lgb_params = {
    "objective": "poisson",
    "metric": ["poisson", "rmse"],
    "boosting_type": "gbdt",
    "learning_rate": LEARNING_RATE,
    "num_leaves": NUM_LEAVES,
    "min_data_in_leaf": MIN_DATA_IN_LEAF,
    "feature_fraction": FEATURE_FRACTION,
    "bagging_fraction": BAGGING_FRACTION,
    "bagging_freq": BAGGING_FREQ,
    "max_bin": MAX_BIN,
    "bin_construct_sample_cnt": BIN_CONSTRUCT_SAMPLE_CNT,
    "feature_penalty": feature_penalty,
    "lambda_l1": LAMBDA_L1,
    "min_gain_to_split": MIN_GAIN_TO_SPLIT,
    "force_col_wise": True,
    "num_threads": NUM_THREADS,
    "seed": SEED,
    "verbosity": -1
}

# Log only safe params. Do not log feature_penalty.
safe_log_params({k: v for k, v in lgb_params.items() if k not in DO_NOT_LOG_PARAM_KEYS}, prefix="lgbm_")

evals_result = {}
start_time = time.time()

booster = lgb.train(
    params=lgb_params,
    train_set=train_data,
    num_boost_round=NUM_ITERATIONS,
    valid_sets=[train_data, test_data],
    valid_names=["train", "test"],
    callbacks=[
        lgb.log_evaluation(period=25),
        lgb.record_evaluation(evals_result),
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS)
    ]
)

elapsed = time.time() - start_time

print("===================================================")
print("TRAINING COMPLETE")
print("===================================================")
print(f"Elapsed seconds: {elapsed:,.2f}")
print("===================================================")

safe_log_metrics({
    "training_elapsed_seconds": elapsed,
    "best_iteration": booster.best_iteration if booster.best_iteration is not None else booster.current_iteration(),
    "num_trees": booster.num_trees(),
    "current_iteration": booster.current_iteration()
})

# Log eval metrics by iteration.
for split_name, metric_dict in evals_result.items():
    for metric_name, values in metric_dict.items():
        for step, value in enumerate(values, start=1):
            try:
                mlflow.log_metric(f"{split_name}_{metric_name}", float(value), step=step)
            except Exception:
                pass

save_json_to_output(
    evals_result,
    "lightgbm_evals_result_wo_segment_new_P12_fts.json",
    log_to_mlflow=True,
    artifact_path="training"
)

model_filename = "lightgbm_nbrx_model_wo_segment_new_P12_fts.txt"
local_model_path = os.path.join(LOCAL_OUTPUT_DIR, model_filename)
booster.save_model(local_model_path)

copy_local_to_output(local_model_path, model_filename)
log_artifact_if_active(local_model_path, artifact_path="model_file")

try:
    mlflow.lightgbm.log_model(booster, artifact_path="model")
    print("MLflow LightGBM model logged.")
except Exception as e:
    print(f"MLflow LightGBM model logging skipped/failed: {e}")

## Cell 14 - Learning curve plots

In [0]:
if "train" in evals_result and "test" in evals_result:
    metrics_available = sorted(set(evals_result["train"].keys()) & set(evals_result["test"].keys()))

    for metric_name in metrics_available:
        plt.figure(figsize=(10, 6))
        plt.plot(evals_result["train"][metric_name], label=f"train {metric_name}", linewidth=2)
        plt.plot(evals_result["test"][metric_name], label=f"test {metric_name}", linewidth=2)
        plt.xlabel("Iteration")
        plt.ylabel(metric_name)
        plt.title(f"LightGBM {metric_name} over iterations")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        save_current_fig(
            f"learning_curve_{metric_name}_wo_segment_new_P12_fts.png",
            log_to_mlflow=True,
            artifact_path="training_plots"
        )

        plt.show()
        plt.close()

## Cell 15 - Predictions and train/test metrics and Lift Chart

In [0]:
start_time = time.time()

pred_train = booster.predict(X_train, num_threads=NUM_THREADS)
pred_test = booster.predict(X_test, num_threads=NUM_THREADS)

elapsed = time.time() - start_time
print("Prediction complete.")
print(f"Elapsed seconds: {elapsed:,.2f}")
safe_log_metrics({"prediction_elapsed_seconds": elapsed})


def calculate_metrics(y_true, y_pred, dataset_name):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    valid_mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true_valid = y_true[valid_mask]
    y_pred_valid = y_pred[valid_mask]

    r2 = r2_score(y_true_valid, y_pred_valid)
    rmse = float(np.sqrt(np.mean((y_true_valid - y_pred_valid) ** 2)))
    mae = float(np.mean(np.abs(y_true_valid - y_pred_valid)))

    nonzero_mask = y_true_valid != 0
    if nonzero_mask.sum() > 0:
        mape = float(np.mean(np.abs((y_true_valid[nonzero_mask] - y_pred_valid[nonzero_mask]) / y_true_valid[nonzero_mask])) * 100.0)
    else:
        mape = np.nan

    pred_positive = np.clip(y_pred_valid, 1e-12, None)
    poisson_nll = float(np.mean(pred_positive - y_true_valid * np.log(pred_positive)))

    return {
        "dataset": dataset_name,
        "r2": float(r2),
        "rmse": rmse,
        "mae": mae,
        "mape_pct": float(mape) if np.isfinite(mape) else np.nan,
        "poisson_nll_without_constant": poisson_nll,
        "rows_total": int(len(y_true)),
        "rows_used_for_r2": int(valid_mask.sum()),
        "rows_used_for_mape": int(nonzero_mask.sum()),
        "zero_target_rows_excluded_from_mape": int((y_true_valid == 0).sum())
    }


train_metrics = calculate_metrics(y_train, pred_train, "train")
test_metrics = calculate_metrics(y_test, pred_test, "test")
metrics_df = pd.DataFrame([train_metrics, test_metrics])

save_df_to_output(
    metrics_df,
    "train_test_metrics_wo_segment_new_P12_fts.csv",
    log_to_mlflow=True,
    artifact_path="metrics"
)

for row in metrics_df.to_dict(orient="records"):
    dataset_name = row["dataset"]
    safe_log_metrics({
        "r2": row["r2"],
        "rmse": row["rmse"],
        "mae": row["mae"],
        "mape_pct": row["mape_pct"],
        "poisson_nll_without_constant": row["poisson_nll_without_constant"],
        "rows_total": row["rows_total"],
        "rows_used_for_mape": row["rows_used_for_mape"],
        "zero_target_rows_excluded_from_mape": row["zero_target_rows_excluded_from_mape"]
    }, prefix=f"{dataset_name}_")

display(metrics_df)

In [0]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# Lift Chart - Test Data
# ============================================================

start_time = time.time()

OUTPUT_SUFFIX = "wo_segment_new_P12_fts"
ARTIFACT_PATH = "lift_charts"
DATASET_NAME = "test"
N_BINS = 10

# ------------------------------------------------------------
# Inputs expected from previous cells:
# y_test
# pred_test
# save_current_fig()
# save_df_to_output()
# safe_log_metrics()
# ------------------------------------------------------------

y_true = np.asarray(y_test).reshape(-1)
y_pred = np.asarray(pred_test).reshape(-1)

if len(y_true) != len(y_pred):
    raise ValueError(
        f"Length mismatch: y_test has {len(y_true)} rows, "
        f"but pred_test has {len(y_pred)} rows."
    )

# Keep valid records only
valid_mask = np.isfinite(y_true) & np.isfinite(y_pred)

actual = y_true[valid_mask]
predicted = y_pred[valid_mask]

if len(actual) == 0:
    raise ValueError("No valid rows available for lift chart.")

overall_actual_mean = float(np.mean(actual))
overall_predicted_mean = float(np.mean(predicted))

if np.isclose(overall_actual_mean, 0):
    raise ValueError("Overall actual mean is 0, so actual lift cannot be calculated.")

if np.isclose(overall_predicted_mean, 0):
    raise ValueError("Overall predicted mean is 0, so predicted lift cannot be calculated.")

# ------------------------------------------------------------
# Sort records by prediction in descending order
# Top predicted records will appear at decile 10 on the chart
# ------------------------------------------------------------

sort_idx = np.argsort(-predicted, kind="mergesort")

actual_sorted = actual[sort_idx]
predicted_sorted = predicted[sort_idx]

n_rows = len(actual_sorted)

# Create decile buckets
bucket_idx = np.floor(np.arange(n_rows) * N_BINS / n_rows).astype(int)
bucket_idx = np.clip(bucket_idx, 0, N_BINS - 1)

# ------------------------------------------------------------
# Fast aggregation
# ------------------------------------------------------------

records = np.bincount(bucket_idx, minlength=N_BINS)

actual_sum = np.bincount(
    bucket_idx,
    weights=actual_sorted,
    minlength=N_BINS
)

predicted_sum = np.bincount(
    bucket_idx,
    weights=predicted_sorted,
    minlength=N_BINS
)

actual_mean = actual_sum / records
predicted_mean = predicted_sum / records

actual_lift = actual_mean / overall_actual_mean
predicted_lift = predicted_mean / overall_predicted_mean

decile_labels = np.arange(10, 101, 10)

lift_df = pd.DataFrame({
    "decile": decile_labels,
    "records": records.astype(int),
    "actual_mean": actual_mean,
    "predicted_mean": predicted_mean,
    "actual_lift": actual_lift,
    "predicted_lift": predicted_lift,
    "actual_sum": actual_sum,
    "predicted_sum": predicted_sum
})

total_actual = float(lift_df["actual_sum"].sum())

if total_actual > 0:
    lift_df["actual_capture_pct"] = lift_df["actual_sum"] / total_actual * 100
    lift_df["cumulative_actual_capture_pct"] = lift_df["actual_capture_pct"].cumsum()
else:
    lift_df["actual_capture_pct"] = np.nan
    lift_df["cumulative_actual_capture_pct"] = np.nan

# ------------------------------------------------------------
# Plot chart
# ------------------------------------------------------------

plt.figure(figsize=(10, 7))

plt.plot(
    lift_df["decile"],
    lift_df["predicted_lift"],
    marker="o",
    linewidth=2,
    label="Predicted lift"
)

plt.plot(
    lift_df["decile"],
    lift_df["actual_lift"],
    marker="s",
    linestyle="--",
    linewidth=2,
    label="Actual lift"
)

plt.xlabel("Decile (ranked by Predicted NBRx)", fontsize=12)
plt.ylabel("Per-decile lift (mean / overall mean)", fontsize=12)
plt.title("Lift Chart", fontsize=16)

plt.xticks(decile_labels)
plt.ylim(bottom=0)

plt.grid(True, alpha=0.25)
plt.legend(fontsize=11)
plt.tight_layout()

# ------------------------------------------------------------
# Save chart to artifact
# ------------------------------------------------------------

save_current_fig(
    f"lift_chart_{DATASET_NAME}_{OUTPUT_SUFFIX}.png",
    log_to_mlflow=True,
    artifact_path=ARTIFACT_PATH
)

plt.show()
plt.close()

# ------------------------------------------------------------
# Save lift table to artifact
# ------------------------------------------------------------

save_df_to_output(
    lift_df,
    f"lift_decile_summary_{DATASET_NAME}_{OUTPUT_SUFFIX}.csv",
    log_to_mlflow=True,
    artifact_path=ARTIFACT_PATH
)

# ------------------------------------------------------------
# Log lift metrics
# ------------------------------------------------------------

elapsed = time.time() - start_time

safe_log_metrics({
    "lift_chart_elapsed_seconds": float(elapsed),
    "lift_rows_total": int(len(y_true)),
    "lift_rows_used": int(len(actual)),
    "lift_overall_actual_mean": float(overall_actual_mean),
    "lift_overall_predicted_mean": float(overall_predicted_mean),
    "lift_top_decile_actual": float(lift_df.loc[0, "actual_lift"]),
    "lift_top_decile_predicted": float(lift_df.loc[0, "predicted_lift"]),
    "lift_top_decile_actual_mean": float(lift_df.loc[0, "actual_mean"]),
    "lift_top_decile_predicted_mean": float(lift_df.loc[0, "predicted_mean"]),
    "lift_top_decile_actual_capture_pct": float(lift_df.loc[0, "actual_capture_pct"])
}, prefix=f"{DATASET_NAME}_")

print("Lift chart completed and saved to artifact.")
print(f"Artifact path: {ARTIFACT_PATH}")
print(f"Chart file: lift_chart_{DATASET_NAME}_{OUTPUT_SUFFIX}.png")
print(f"Summary file: lift_decile_summary_{DATASET_NAME}_{OUTPUT_SUFFIX}.csv")
print(f"Rows used: {len(actual):,}")
print(f"Top decile actual lift: {lift_df.loc[0, 'actual_lift']:.4f}")
print(f"Top decile predicted lift: {lift_df.loc[0, 'predicted_lift']:.4f}")
print(f"Elapsed seconds: {elapsed:,.2f}")

display(lift_df)

## Cell 16 - Feature importance and top 1000 by gain

In [0]:
gain_importance = booster.feature_importance(importance_type="gain")
split_importance = booster.feature_importance(importance_type="split")

importance_df = pd.DataFrame({
    "feature_index": np.arange(n_features, dtype=np.int32),
    "feature_alias": feature_aliases,
    "feature": feature_cols,
    "gain_importance": gain_importance.astype(float),
    "split_importance": split_importance.astype(float)
})

importance_df = importance_df.sort_values(by="gain_importance", ascending=False).reset_index(drop=True)
importance_df["gain_rank"] = np.arange(1, len(importance_df) + 1)

save_df_to_output(
    importance_df,
    "all_feature_importance_gain_wo_segment_new_P12_fts.csv",
    log_to_mlflow=True,
    artifact_path="feature_importance"
)

top_k_effective = min(TOP_K, len(importance_df))

top_features_gain_df = importance_df.head(top_k_effective).copy().reset_index(drop=True)

top_feature_indices_gain = top_features_gain_df["feature_index"].to_numpy(dtype=np.int32)
top_feature_aliases_gain = top_features_gain_df["feature_alias"].tolist()
top_feature_names_gain = top_features_gain_df["feature"].tolist()

save_df_to_output(
    top_features_gain_df,
    "top_1000_features_gain_only_wo_segment_new_P12_fts.csv",
    log_to_mlflow=True,
    artifact_path="feature_importance"
)

save_text_list_to_output(
    top_feature_names_gain,
    "top_1000_feature_names_gain_only_wo_segment_new_P12_fts.txt",
    log_to_mlflow=True,
    artifact_path="feature_importance"
)

print("Top gain features selected:", len(top_features_gain_df))
print("First 20 gain-based features:")
print(top_feature_names_gain[:20])

display(top_features_gain_df.head(50))

## Cell 17 - SHAP aggregation for all features on train rows

In [0]:
print("===================================================")
print("STARTING SHAP FOR ALL FEATURES")
print("===================================================")
print("Train rows:", X_train.shape[0])
print("Features  :", n_features)
print("SHAP batch size:", SHAP_BATCH_SIZE)
print("===================================================")

shap_sum = np.zeros(n_features, dtype=np.float64)
shap_abs_sum = np.zeros(n_features, dtype=np.float64)
total_shap_rows = 0

start_time = time.time()

for start in range(0, X_train.shape[0], SHAP_BATCH_SIZE):
    end = min(start + SHAP_BATCH_SIZE, X_train.shape[0])
    X_batch = X_train[start:end, :]

    shap_batch_all = booster.predict(
        X_batch,
        pred_contrib=True,
        num_threads=NUM_THREADS
    )

    shap_batch = shap_batch_all[:, :n_features]

    shap_sum += shap_batch.sum(axis=0)
    shap_abs_sum += np.abs(shap_batch).sum(axis=0)
    total_shap_rows += end - start

    if total_shap_rows % (SHAP_BATCH_SIZE * 10) == 0 or end == X_train.shape[0]:
        elapsed = time.time() - start_time
        print(f"Processed SHAP rows: {total_shap_rows:,} / {X_train.shape[0]:,} | elapsed sec: {elapsed:,.1f}")

    del X_batch, shap_batch_all, shap_batch
    gc.collect()

elapsed = time.time() - start_time
print("===================================================")
print("SHAP COMPLETE")
print("===================================================")
print("Rows used for SHAP:", total_shap_rows)
print(f"Elapsed seconds   : {elapsed:,.2f}")
print("===================================================")

safe_log_metrics({
    "shap_aggregation_elapsed_seconds": elapsed,
    "shap_aggregation_rows": total_shap_rows
})

## Cell 18 - SHAP ranking and selected feature outputs

In [0]:
import os
import numpy as np
import pandas as pd


# ============================================================
# Bucket mapping setup
# ============================================================

BUCKET_MAPPING_FILE_NAME = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/Bucket_mapping.csv"

# Keep this as None if Bucket_mapping.csv is in the same folder as this code/notebook.
# If Databricks cannot find the file, replace None with full path, for example:
# BUCKET_MAPPING_PATH = "/Workspace/Users/your_folder/Bucket_mapping.csv"
# or
# BUCKET_MAPPING_PATH = "/dbfs/FileStore/your_folder/Bucket_mapping.csv"
BUCKET_MAPPING_PATH = None


def find_bucket_mapping_file(file_name="/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_oe_journey/kerendia_oe_volume/oe_git_offload/Kerendia_OE_2.0/Feature Selection/Bucket_mapping.csv", explicit_path=None):
    candidates = []

    if explicit_path:
        candidates.append(explicit_path)

    # Script / repo execution path
    if "__file__" in globals():
        candidates.append(os.path.join(os.path.dirname(os.path.abspath(__file__)), file_name))

    # Current working directory
    candidates.append(os.path.join(os.getcwd(), file_name))
    candidates.append(file_name)

    # Databricks notebook workspace folder path
    try:
        notebook_path = (
            dbutils
            .notebook
            .entry_point
            .getDbutils()
            .notebook()
            .getContext()
            .notebookPath()
            .get()
        )
        notebook_dir = os.path.dirname(notebook_path)
        candidates.append(f"/Workspace{notebook_dir}/{file_name}")
    except Exception:
        pass

    # Remove duplicate candidates while preserving order
    seen = set()
    unique_candidates = []
    for path in candidates:
        if path and path not in seen:
            unique_candidates.append(path)
            seen.add(path)

    for path in unique_candidates:
        if os.path.exists(path):
            return path

    raise FileNotFoundError(
        "Bucket_mapping.csv was not found. Checked these paths:\n"
        + "\n".join(unique_candidates)
        + "\n\nSet BUCKET_MAPPING_PATH manually if needed."
    )


bucket_mapping_path = find_bucket_mapping_file(
    file_name=BUCKET_MAPPING_FILE_NAME,
    explicit_path=BUCKET_MAPPING_PATH
)

bucket_mapping_df = pd.read_csv(bucket_mapping_path)

required_bucket_cols = {"Feature", "Bucket"}
missing_bucket_cols = required_bucket_cols - set(bucket_mapping_df.columns)

if missing_bucket_cols:
    raise ValueError(
        f"Bucket_mapping.csv is missing required columns: {missing_bucket_cols}. "
        f"Available columns are: {list(bucket_mapping_df.columns)}"
    )

bucket_mapping_df = bucket_mapping_df[["Feature", "Bucket"]].copy()

bucket_mapping_df["Feature"] = bucket_mapping_df["Feature"].astype("string").str.strip()
bucket_mapping_df["Bucket"] = bucket_mapping_df["Bucket"].astype("string").str.strip()

bucket_mapping_df = bucket_mapping_df.dropna(subset=["Feature"])
bucket_mapping_df = bucket_mapping_df[bucket_mapping_df["Feature"] != ""]

duplicate_feature_count = bucket_mapping_df["Feature"].duplicated().sum()

if duplicate_feature_count > 0:
    print(f"Duplicate features found in Bucket_mapping.csv: {duplicate_feature_count}")
    print("Keeping the first bucket mapping for each feature.")

bucket_mapping_df = bucket_mapping_df.drop_duplicates(subset=["Feature"], keep="first")

feature_to_bucket = bucket_mapping_df.set_index("Feature")["Bucket"]

print(f"Bucket mapping loaded from: {bucket_mapping_path}")
print(f"Bucket mapping rows used: {len(bucket_mapping_df):,}")


# ============================================================
# SHAP ranking with bucket
# ============================================================

mean_shap_train = shap_sum / total_shap_rows
mean_abs_shap_train = shap_abs_sum / total_shap_rows

total_mean_abs_shap = float(np.sum(mean_abs_shap_train))

if total_mean_abs_shap > 0:
    percent_impact_shap = 100.0 * mean_abs_shap_train / total_mean_abs_shap
else:
    percent_impact_shap = np.zeros_like(mean_abs_shap_train)

shap_rank_df = pd.DataFrame({
    "feature_index": np.arange(n_features, dtype=np.int32),
    "feature_alias": feature_aliases,
    "feature": feature_cols,
    "mean_shap_train": mean_shap_train,
    "mean_abs_shap_train": mean_abs_shap_train,
    "percent_impact_shap": percent_impact_shap
})

# ------------------------------------------------------------
# Add bucket using feature name
# Fallback: if not found on feature, try feature_alias
# ------------------------------------------------------------

shap_rank_df["feature_key_for_bucket"] = shap_rank_df["feature"].astype("string").str.strip()
shap_rank_df["bucket"] = shap_rank_df["feature_key_for_bucket"].map(feature_to_bucket)

missing_bucket_mask = shap_rank_df["bucket"].isna()

if missing_bucket_mask.any():
    alias_key_for_bucket = shap_rank_df.loc[missing_bucket_mask, "feature_alias"].astype("string").str.strip()
    shap_rank_df.loc[missing_bucket_mask, "bucket"] = alias_key_for_bucket.map(feature_to_bucket)

shap_rank_df["bucket"] = shap_rank_df["bucket"].fillna("Unmapped")
shap_rank_df = shap_rank_df.drop(columns=["feature_key_for_bucket"])

shap_rank_df = shap_rank_df.sort_values(
    by="mean_abs_shap_train",
    ascending=False
).reset_index(drop=True)

shap_rank_df["shap_rank"] = np.arange(1, len(shap_rank_df) + 1)
shap_rank_df["cumulative_percent_impact_shap"] = shap_rank_df["percent_impact_shap"].cumsum()

shap_rank_df = shap_rank_df[[
    "shap_rank",
    "feature_index",
    "feature_alias",
    "feature",
    "bucket",
    "mean_shap_train",
    "mean_abs_shap_train",
    "percent_impact_shap",
    "cumulative_percent_impact_shap"
]]


# ============================================================
# Save all SHAP ranking outputs with bucket
# ============================================================

save_df_to_output(
    shap_rank_df,
    "shap_ranking_all_features_all_train_rows_wo_segment_new_P12_fts.csv",
    log_to_mlflow=True,
    artifact_path="shap_outputs"
)

top_100_shap_df = shap_rank_df.head(min(100, len(shap_rank_df))).copy().reset_index(drop=True)
top_200_shap_df = shap_rank_df.head(min(200, len(shap_rank_df))).copy().reset_index(drop=True)
top_500_shap_df = shap_rank_df.head(min(500, len(shap_rank_df))).copy().reset_index(drop=True)
top_1000_shap_df = shap_rank_df.head(min(1000, len(shap_rank_df))).copy().reset_index(drop=True)

for name, df_temp in [
    ("top_100_features_by_shap_wo_segment_new_P12_fts.csv", top_100_shap_df),
    ("top_200_features_by_shap_wo_segment_new_P12_fts.csv", top_200_shap_df),
    ("top_500_features_by_shap_wo_segment_new_P12_fts.csv", top_500_shap_df),
    ("top_1000_features_by_shap_wo_segment_new_P12_fts.csv", top_1000_shap_df),
]:
    save_df_to_output(
        df_temp,
        name,
        log_to_mlflow=True,
        artifact_path="shap_outputs"
    )

save_text_list_to_output(
    top_1000_shap_df["feature"].tolist(),
    "top_1000_feature_names_by_shap_wo_segment_new_P12_fts.txt",
    log_to_mlflow=True,
    artifact_path="shap_outputs"
)


# ============================================================
# Content feature outputs with bucket
# ============================================================

content_all_shap_df = shap_rank_df[
    shap_rank_df["feature"].apply(is_content_feature)
].copy().reset_index(drop=True)

content_top_1000_shap_df = top_1000_shap_df[
    top_1000_shap_df["feature"].apply(is_content_feature)
].copy().reset_index(drop=True)

save_df_to_output(
    content_all_shap_df,
    "content_features_all_by_shap_wo_segment_new_P12_fts.csv",
    log_to_mlflow=True,
    artifact_path="content_features"
)

save_df_to_output(
    content_top_1000_shap_df,
    "content_features_in_top_1000_by_shap_wo_segment_new_P12_fts.csv",
    log_to_mlflow=True,
    artifact_path="content_features"
)


# ============================================================
# Digital features in Top 1000 SHAP features
# Artifact path: Digital_in_top_1k
# No cumulative_percent_impact_shap column in this output
# ============================================================

digital_top_1000_shap_df = top_1000_shap_df[
    top_1000_shap_df["bucket"].astype("string").str.strip().str.lower() == "digital"
].copy().reset_index(drop=True)

digital_top_1000_shap_df = digital_top_1000_shap_df.drop(
    columns=["cumulative_percent_impact_shap"],
    errors="ignore"
)

digital_top_1000_shap_df = digital_top_1000_shap_df[[
    "shap_rank",
    "feature_index",
    "feature_alias",
    "feature",
    "bucket",
    "mean_shap_train",
    "mean_abs_shap_train",
    "percent_impact_shap"
]]

save_df_to_output(
    digital_top_1000_shap_df,
    "digital_features_in_top_1000_by_shap_wo_segment_new_P12_fts.csv",
    log_to_mlflow=True,
    artifact_path="Digital_in_top_1k"
)


# ============================================================
# Final selected output with bucket
# ============================================================

final_feature_selection_summary_df = pd.DataFrame({
    "selected_feature_rank": np.arange(1, len(top_1000_shap_df) + 1),
    "shap_rank": top_1000_shap_df["shap_rank"].tolist(),
    "feature_index": top_1000_shap_df["feature_index"].tolist(),
    "feature_alias": top_1000_shap_df["feature_alias"].tolist(),
    "feature": top_1000_shap_df["feature"].tolist(),
    "bucket": top_1000_shap_df["bucket"].tolist(),
    "mean_shap_train": top_1000_shap_df["mean_shap_train"].tolist(),
    "mean_abs_shap_train": top_1000_shap_df["mean_abs_shap_train"].tolist(),
    "percent_impact_shap": top_1000_shap_df["percent_impact_shap"].tolist()
})

save_df_to_output(
    final_feature_selection_summary_df,
    "final_selected_top_1000_features_by_shap_wo_segment_new_P12_fts.csv",
    log_to_mlflow=True,
    artifact_path="final_outputs"
)


# ============================================================
# Metrics logging
# ============================================================

mapped_bucket_count = int((shap_rank_df["bucket"] != "Unmapped").sum())
unmapped_bucket_count = int((shap_rank_df["bucket"] == "Unmapped").sum())

safe_log_metrics({
    "bucket_mapping_rows_used": len(bucket_mapping_df),
    "features_with_bucket_mapping": mapped_bucket_count,
    "features_without_bucket_mapping": unmapped_bucket_count,
    "content_feature_count_all": len(content_all_shap_df),
    "content_feature_count_in_top_1000": len(content_top_1000_shap_df),
    "digital_feature_count_in_top_1000": len(digital_top_1000_shap_df)
})

print("Top 1000 SHAP features:", len(top_1000_shap_df))
print("All content features:", len(content_all_shap_df))
print("Content features in top 1000:", len(content_top_1000_shap_df))
print("Digital features in top 1000:", len(digital_top_1000_shap_df))
print("Features with bucket mapping:", mapped_bucket_count)
print("Features without bucket mapping:", unmapped_bucket_count)

display(shap_rank_df.head(20))

## Cell 19 - Save HCP-level SHAP CSV parts
This is saved to CSV parts only and is not logged to MLflow. It includes `PROVIDER_ID`, `dataset_split`, target, prediction, expected value, and SHAP columns. By default it writes top-1000 SHAP features.

In [0]:
if HCP_SHAP_USE_ALL_FEATURES:
    HCP_SHAP_FEATURE_DF = shap_rank_df.copy()
    hcp_feature_set_name = "all_features"
else:
    HCP_SHAP_FEATURE_DF = top_1000_shap_df.copy()
    hcp_feature_set_name = "top_1000_by_shap"

HCP_SHAP_OUTPUT_SUBDIR = f"hcp_level_shap_values_{hcp_feature_set_name}_wo_segment_new_P12_fts"

try:
    dbutils.fs.mkdirs(f"dbfs:{OUTPUT_DIR.rstrip('/')}/{HCP_SHAP_OUTPUT_SUBDIR}")
except Exception:
    Path(f"{OUTPUT_DIR.rstrip('/')}/{HCP_SHAP_OUTPUT_SUBDIR}").mkdir(parents=True, exist_ok=True)


def write_hcp_level_shap_csv_parts(
    booster,
    X,
    y,
    preds,
    provider_ids,
    split_name,
    feature_df,
    batch_size,
    num_threads,
    output_subdir
):
    feature_indices = feature_df["feature_index"].to_numpy(dtype=np.int32)
    feature_names = feature_df["feature"].tolist()
    shap_col_names = ["SHAP__" + str(f) for f in feature_names]

    n_rows_local = X.shape[0]
    part_num = 0
    start_time = time.time()

    print("===================================================")
    print(f"Writing HCP SHAP CSV parts for split: {split_name}")
    print("Rows:", n_rows_local)
    print("Features:", len(feature_indices))
    print("===================================================")

    for start in range(0, n_rows_local, batch_size):
        end = min(start + batch_size, n_rows_local)
        X_batch = X[start:end, :]

        shap_batch_all = booster.predict(
            X_batch,
            pred_contrib=True,
            num_threads=num_threads
        )

        shap_selected = shap_batch_all[:, feature_indices].astype(np.float32)
        expected_value = shap_batch_all[:, -1].astype(np.float32)

        out_df = pd.DataFrame(shap_selected, columns=shap_col_names)
        out_df.insert(0, ID_COL, provider_ids[start:end])
        out_df.insert(1, "dataset_split", split_name)
        out_df.insert(2, TARGET_COL, y[start:end])
        out_df.insert(3, "prediction", preds[start:end])
        out_df.insert(4, "expected_value", expected_value)

        local_file = os.path.join(LOCAL_OUTPUT_DIR, f"hcp_shap_{split_name}_part_{part_num:05d}.csv")
        out_df.to_csv(local_file, index=False)

        # Save to volume, do not log to MLflow.
        copy_local_to_output(local_file, f"{output_subdir}/hcp_shap_{split_name}_part_{part_num:05d}.csv")

        elapsed = time.time() - start_time
        print(f"{split_name} rows written: {end:,} / {n_rows_local:,} | part {part_num} | elapsed sec: {elapsed:,.1f}")

        part_num += 1
        del X_batch, shap_batch_all, shap_selected, expected_value, out_df
        gc.collect()

    print(f"Completed HCP-level SHAP CSV for split: {split_name}")


write_hcp_level_shap_csv_parts(
    booster=booster,
    X=X_train,
    y=y_train,
    preds=pred_train,
    provider_ids=provider_train,
    split_name="train",
    feature_df=HCP_SHAP_FEATURE_DF,
    batch_size=HCP_SHAP_BATCH_SIZE,
    num_threads=NUM_THREADS,
    output_subdir=HCP_SHAP_OUTPUT_SUBDIR
)

write_hcp_level_shap_csv_parts(
    booster=booster,
    X=X_test,
    y=y_test,
    preds=pred_test,
    provider_ids=provider_test,
    split_name="test",
    feature_df=HCP_SHAP_FEATURE_DF,
    batch_size=HCP_SHAP_BATCH_SIZE,
    num_threads=NUM_THREADS,
    output_subdir=HCP_SHAP_OUTPUT_SUBDIR
)

print("HCP-level SHAP CSV parts saved here:")
print(f"{OUTPUT_DIR.rstrip('/')}/{HCP_SHAP_OUTPUT_SUBDIR}")

## Cell 20 - SHAP beeswarm and mean absolute SHAP plots for requested feature sets

In [0]:
rng = np.random.default_rng(SEED)
plot_n = min(PLOT_ROW_SAMPLE, X_train.shape[0])
plot_row_indices = np.sort(rng.choice(X_train.shape[0], size=plot_n, replace=False))

plot_feature_sets = {
    "top_100_features": top_100_shap_df,
    "top_200_features": top_200_shap_df,
    "top_500_features": top_500_shap_df,
    "top_1000_features": top_1000_shap_df,
    "all_content_features": content_all_shap_df,
    "content_features_in_top_1000": content_top_1000_shap_df
}

plot_feature_sets = {k: v for k, v in plot_feature_sets.items() if v is not None and len(v) > 0}

plot_union_indices = sorted(set(idx for df_temp in plot_feature_sets.values() for idx in df_temp["feature_index"].astype(int).tolist()))
union_index_to_position = {idx: pos for pos, idx in enumerate(plot_union_indices)}

print("Rows used for SHAP plots:", len(plot_row_indices))
print("Union feature count for plots:", len(plot_union_indices))


def collect_shap_for_union_features(booster, X, row_indices, feature_indices, batch_size, num_threads):
    shap_blocks = []
    feature_value_blocks = []
    total_rows = len(row_indices)
    rows_done = 0
    start_time = time.time()

    for start in range(0, total_rows, batch_size):
        end = min(start + batch_size, total_rows)
        batch_rows = row_indices[start:end]
        X_batch_full = X[batch_rows, :]

        shap_batch_all = booster.predict(
            X_batch_full,
            pred_contrib=True,
            num_threads=num_threads
        )

        shap_batch_selected = shap_batch_all[:, feature_indices]
        X_batch_selected = X_batch_full[:, feature_indices]

        shap_blocks.append(shap_batch_selected.astype(np.float32))
        feature_value_blocks.append(X_batch_selected.astype(np.float32))

        rows_done += end - start
        if rows_done % (batch_size * 10) == 0 or rows_done == total_rows:
            elapsed = time.time() - start_time
            print(f"Collected SHAP plot rows: {rows_done:,} / {total_rows:,} | elapsed sec: {elapsed:,.1f}")

        del X_batch_full, shap_batch_all, shap_batch_selected, X_batch_selected
        gc.collect()

    return np.vstack(shap_blocks), np.vstack(feature_value_blocks)


shap_plot_union, X_plot_union = collect_shap_for_union_features(
    booster=booster,
    X=X_train,
    row_indices=plot_row_indices,
    feature_indices=plot_union_indices,
    batch_size=SHAP_BATCH_SIZE,
    num_threads=NUM_THREADS
)


def create_beeswarm_plot_for_feature_set(set_name, feature_df):
    feature_indices = feature_df["feature_index"].astype(int).tolist()
    feature_names = feature_df["feature"].tolist()
    positions = [union_index_to_position[idx] for idx in feature_indices]

    shap_values = shap_plot_union[:, positions]
    X_values = X_plot_union[:, positions]
    X_values_df = pd.DataFrame(X_values, columns=feature_names)

    n_display = len(feature_names)
    plot_height = min(max(8, 0.25 * n_display), 110)

    plt.figure()
    shap.summary_plot(
        shap_values,
        X_values_df,
        feature_names=feature_names,
        max_display=n_display,
        plot_type="dot",
        show=False,
        plot_size=(16, plot_height)
    )
    plt.tight_layout()

    filename = f"shap_beeswarm_{set_name}.pdf"
    local_path = os.path.join(LOCAL_OUTPUT_DIR, filename)
    plt.savefig(local_path, format="pdf", bbox_inches="tight")

    copy_local_to_output(local_path, f"shap_plots/{filename}")
    log_artifact_if_active(local_path, artifact_path="shap_plots")

    plt.show()
    plt.close()


def create_abs_shap_plot_for_feature_set(set_name, feature_df):
    plot_df = feature_df.copy().sort_values("mean_abs_shap_train", ascending=True)
    n_display = len(plot_df)
    plot_height = min(max(8, 0.25 * n_display), 110)

    plt.figure(figsize=(16, plot_height))
    plt.barh(plot_df["feature"], plot_df["mean_abs_shap_train"])
    plt.xlabel("mean(|SHAP value|) on train data")
    plt.ylabel("Feature")
    plt.title(f"Mean Absolute SHAP - {set_name}")
    plt.tight_layout()

    filename = f"mean_abs_shap_{set_name}.pdf"
    local_path = os.path.join(LOCAL_OUTPUT_DIR, filename)
    plt.savefig(local_path, format="pdf", bbox_inches="tight")

    copy_local_to_output(local_path, f"shap_plots/{filename}")
    log_artifact_if_active(local_path, artifact_path="shap_plots")

    plt.show()
    plt.close()


for set_name, feature_df in plot_feature_sets.items():
    print("===================================================")
    print("Creating SHAP plots for:", set_name)
    print("Feature count:", len(feature_df))
    print("===================================================")

    create_beeswarm_plot_for_feature_set(set_name, feature_df)
    create_abs_shap_plot_for_feature_set(set_name, feature_df)

## Cell 21 - PDP plots for content features only

In [0]:
content_pdp_feature_df = content_all_shap_df.copy()

if CONTENT_PDP_MAX_FEATURES is not None:
    content_pdp_feature_df = content_pdp_feature_df.head(CONTENT_PDP_MAX_FEATURES).copy()

content_pdp_features = content_pdp_feature_df["feature"].tolist()
content_pdp_indices = content_pdp_feature_df["feature_index"].astype(int).tolist()

print("Content PDP feature count:", len(content_pdp_features))

if len(content_pdp_features) == 0:
    print("No content features found. Skipping PDP.")
else:
    if CONTENT_PDP_ROWS is None or CONTENT_PDP_ROWS >= X_train.shape[0]:
        pdp_row_indices = np.arange(X_train.shape[0])
    else:
        rng = np.random.default_rng(SEED)
        pdp_row_indices = np.sort(rng.choice(X_train.shape[0], size=min(CONTENT_PDP_ROWS, X_train.shape[0]), replace=False))

    print("Rows used for content PDP:", len(pdp_row_indices))


def build_pdp_grid_from_values(values, grid_size=20, lower_q=0.01, upper_q=0.99):
    values = np.asarray(values, dtype=np.float32)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.array([], dtype=np.float32)

    unique_values = np.unique(values)

    if len(unique_values) <= grid_size:
        grid = unique_values
    else:
        q = np.linspace(lower_q, upper_q, grid_size)
        grid = np.nanquantile(values, q)
        grid = np.unique(grid.astype(np.float32))

    if np.nanmin(values) <= 0 <= np.nanmax(values):
        grid = np.unique(np.concatenate([grid, np.array([0.0], dtype=np.float32)]))

    return np.sort(grid.astype(np.float32))


def compute_pdp_for_feature_from_array(booster, X, row_indices, feature_index, grid_values, batch_size, num_threads):
    pred_sum = np.zeros(len(grid_values), dtype=np.float64)
    pred_count = np.zeros(len(grid_values), dtype=np.int64)

    for start in range(0, len(row_indices), batch_size):
        end = min(start + batch_size, len(row_indices))
        batch_rows = row_indices[start:end]

        X_batch = X[batch_rows, :].copy()

        for j, grid_value in enumerate(grid_values):
            X_batch[:, feature_index] = grid_value
            preds = booster.predict(X_batch, num_threads=num_threads)
            finite_mask = np.isfinite(preds)
            pred_sum[j] += float(np.sum(preds[finite_mask]))
            pred_count[j] += int(np.sum(finite_mask))
            del preds

        del X_batch
        gc.collect()

    return pred_sum / pred_count


if len(content_pdp_features) > 0:
    pdp_rows = []
    local_pdp_pdf = os.path.join(LOCAL_OUTPUT_DIR, "pdp_content_features_only.pdf")
    start_time = time.time()

    with PdfPages(local_pdp_pdf) as pdf:
        for feat_num, (feature_name, feature_index) in enumerate(zip(content_pdp_features, content_pdp_indices), start=1):
            print("===================================================")
            print(f"PDP {feat_num}/{len(content_pdp_features)}:", feature_name)
            print("===================================================")

            feature_values = X_train[pdp_row_indices, feature_index]
            grid_values = build_pdp_grid_from_values(
                feature_values,
                grid_size=CONTENT_PDP_GRID_SIZE,
                lower_q=0.01,
                upper_q=0.99
            )

            if len(grid_values) == 0:
                print("Skipping feature with no finite grid:", feature_name)
                continue

            avg_predictions = compute_pdp_for_feature_from_array(
                booster=booster,
                X=X_train,
                row_indices=pdp_row_indices,
                feature_index=feature_index,
                grid_values=grid_values,
                batch_size=CONTENT_PDP_BATCH_SIZE,
                num_threads=NUM_THREADS
            )

            temp_pdp_df = pd.DataFrame({
                "feature": feature_name,
                "feature_index": feature_index,
                "feature_value": grid_values,
                "avg_prediction": avg_predictions,
                "rows_used": len(pdp_row_indices)
            })

            temp_pdp_df["centered_avg_prediction"] = temp_pdp_df["avg_prediction"] - temp_pdp_df["avg_prediction"].mean()
            pdp_rows.append(temp_pdp_df)

            plt.figure(figsize=(10, 6))
            plt.plot(temp_pdp_df["feature_value"], temp_pdp_df["avg_prediction"], marker="o", linewidth=2)
            plt.xlabel(feature_name)
            plt.ylabel(f"Average predicted {TARGET_COL}")
            plt.title(f"PDP Plot: {feature_name}")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            pdf.savefig(bbox_inches="tight")
            plt.close()

            elapsed = time.time() - start_time
            print(f"Elapsed sec: {elapsed:,.1f}")

    content_pdp_df = pd.concat(pdp_rows, ignore_index=True) if len(pdp_rows) > 0 else pd.DataFrame()

    if len(content_pdp_df) > 0:
        save_df_to_output(
            content_pdp_df,
            "pdp_content_features_only_data.csv",
            log_to_mlflow=True,
            artifact_path="pdp_content_features"
        )

    copy_local_to_output(local_pdp_pdf, "pdp_content_features_only.pdf")
    log_artifact_if_active(local_pdp_pdf, artifact_path="pdp_content_features")

    print("Content PDP PDF saved and logged.")

## Cell 22 - End MLflow run

In [0]:
if mlflow.active_run() is not None:
    print("Ending MLflow run:", mlflow.active_run().info.run_id)
    mlflow.end_run()